In [1]:
## Imports
import numpy as np
import pandas as pd
import requests
import json
import re

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# PyTorch 
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models

import matplotlib.pyplot as plt

Pull Fluorescent Protein Database

In [2]:
url = "https://www.fpbase.org/api/proteins/?format=json"

response = requests.get(url)
data = response.json()

print(type(data))
print(len(data))

<class 'list'>
1040


Transform content of Db to Df

In [3]:
rows = []

for protein in data:
    
    states = protein.get("states", [])
    if len(states) == 0:
        continue
    
    state = states[0]
    
    rows.append({
        "name": protein.get("name"),
        "sequence": protein.get("seq"),
        "aggregation_type": protein.get("agg"),
        "excitation": state.get("ex_max"),
        "emission": state.get("em_max"),
        "brightness": state.get("brightness") 
    })

df = pd.DataFrame(rows)
df.head()

,name,sequence,aggregation_type,excitation,emission,brightness
0,10B,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,,513.0,525.0,NaN
1,11,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,,502.0,512.0,NaN
2,(3-F)Tyr-EGFP,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,,484.0,514.0,NaN
3,5B,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,,512.0,524.0,NaN
4,A44-KR,MEGGPALFQSDMTFKIFIDGEVNGQKFTIVADGSSKFPHGDFNAHA...,,397.0,520.0,NaN


Cleanup of the df: first, identify columns with empty lines

In [4]:
df.isna().sum()

name                  0
sequence             44
aggregation_type      0
excitation            5
emission             49
brightness          296
dtype: int64

Drop lines with lacking values for the parameters needed later

In [5]:
df_no_empty_lines = df.dropna(subset=["sequence", "emission", "emission","brightness"])
df_no_empty_lines.shape
df_no_empty_lines.isna().sum()



name                0
sequence            0
aggregation_type    0
excitation          0
emission            0
brightness          0
dtype: int64

Since sequences may contain peptide tags or linkers, those parts are deleted. First, the linker sequences to delete are imported:

In [10]:
linkers_df = pd.read_csv(r'C:\Users\Adrian\Documents\Studium\M1\ai\linkers_list.csv')
linkers_df.head()

,PDB_code,regiom,length,distance,SA,sequence,secondary_structure,description
1ordA_2,1ordA_2,417 - 424,8,11.484,0.260,GRKLWHDL,HHHHHHHH,CARBOXYLYASE
1floB_1,1floB_1,104 - 129,26,40.131,0.607,IIPYYGQKHQSDITDIVSSLQLQFES,EECCSCSCSCCCHHHHHHHHHHHHHC,LIGASELYASEDNA
1nal1_1,1nal1_1,239 - 245,7,10.248,0.241,CNKVIDL,HHHHHHH,LYASE
1dw9A_1,1dw9A_1,79 - 93,15,23.664,0.584,PLRGCIDDRIPTDPT,CCCCCCSSSSCCSCC,LYASE
1di1A_1,1di1A_1,183 - 190,8,19.209,0.494,TDRARLSI,TCCCCCCC,LYASE


In [21]:
linkers_seq = linkers_df.sequence.tolist()
linkers_sec_str = linkers_df.secondary_structure.tolist()
df_no_empty_lines['sequence'] = df_no_empty_lines['sequence'].replace(to_replace=linkers_seq, value='', regex=True)

df_no_empty_lines['sequence'] = df_no_empty_lines['sequence'].replace(to_replace=linkers_sec_str, value='', regex=True)
df_no_linkers = df_no_empty_lines

Sequences differ in lenght, so they are padded by adding zeroes.

In [22]:
print(df_no_linkers['sequence'].map(len).max())

582


In [23]:
df_no_linkers['padded_sequence']=df_no_empty_lines['sequence'].str.ljust(582, '0')
df_padded = df_no_linkers

Now, the t-scales 5-dimensional vector is assigned to amino acids of every sequence.

In [10]:
t_scales_lookup = pd.read_csv(r'C:\Users\Adrian\Documents\Studium\M1\ai\T_scales_table.csv', skipinitialspace=True)
#t_scales_lookup.head() in case somebody wants to see lookup table
dictionnary = {
    row['symbol']: row[['T_1','T_2','T_3','T_4','T_5']].to_numpy()
    for _, row in t_scales_lookup.iterrows()
}
def encoder(sequence, dictionnary):
    return np.array([dictionnary[char] for char in sequence])
#still for sequence since I'll have it pull PDB sequences once I have enough time to run the PDB sequence lookup
df_padded["encoded_sequence"] = df_padded["padded_sequence"].apply(lambda x: encoder(x, dictionnary))
df_clean = df_padded
df_clean.head()

,name,sequence,aggregation_type,excitation,emission,brightness,padded_sequence,encoded_sequence
6,aacuGFP1,MSLSKHGITQEMPTKYHMKGSVNGHEFEIEGVGTGHPYEGTHMAEL...,t,478.0,502.0,22.51,MSLSKHGITQEMPTKYHMKGSVNGHEFEIEGVGTGHPYEGTHMAEL...,"[[4.08, 0.98, 2.34, 1.64, 0.79], [7.44, 0.65, ..."
7,aacuGFP2,MSYSKQGIVQEMKTKYRMEGSVNGHEFTIEGVGTGYPYEGKQMSEL...,t,502.0,513.0,66.67,MSYSKQGIVQEMKTKYRMEGSVNGHEFTIEGVGTGYPYEGKQMSEL...,"[[4.08, 0.98, 2.34, 1.64, 0.79], [7.44, 0.65, ..."
8,AausFP1,MSYGALLFREKIPYVVEMEGDVEGMKFSVRGKGHGDANTGKIEASF...,d,504.0,510.0,164.90,MSYGALLFREKIPYVVEMEGDVEGMKFSVRGKGHGDANTGKIEASF...,"[[4.08, 0.98, 2.34, 1.64, 0.79], [7.44, 0.65, ..."
12,AausGFP,MKIYCEGEKLFKGIVPIQIELNGDVNGHKFSVHGEGQGEAEMGRLT...,d,398.0,503.0,21.17,MKIYCEGEKLFKGIVPIQIELNGDVNGHKFSVHGEGQGEAEMGRLT...,"[[4.08, 0.98, 2.34, 1.64, 0.79], [2.59, 2.34, ..."
13,aceGFP,MSKGAELFTGIVPILIELNGDVNGHKFSVSGEGEGDATYGKLTLKF...,m,480.0,505.0,27.50,MSKGAELFTGIVPILIELNGDVNGHKFSVSGEGEGDATYGKLTLKF...,"[[4.08, 0.98, 2.34, 1.64, 0.79], [7.44, 0.65, ..."


Here the neural network will be established...